# WiDS 2026 | Tri-Survival Stack + Safe Runtime Fix


In [1]:

import os, sys, gc, re, glob, json, warnings, importlib, subprocess
warnings.filterwarnings("ignore")

# ============================================================
# WiDS 2026 | Tri-Survival Stack + Safe Runtime Fix
# ============================================================

def _pkg_version(dist_name: str):
    try:
        from importlib.metadata import version
        return version(dist_name)
    except Exception:
        return None

def _purge_modules(prefixes=("sklearn", "sksurv")):
    doomed = []
    for name in list(sys.modules):
        for p in prefixes:
            if name == p or name.startswith(p + "."):
                doomed.append(name)
                break
    for name in doomed:
        sys.modules.pop(name, None)
    gc.collect()
    importlib.invalidate_caches()

def _pip_install(pkgs):
    cmd = [sys.executable, "-m", "pip", "install", "-q", "--no-cache-dir", "--upgrade", *pkgs]
    print("[SETUP] pip install:", " ".join(pkgs))
    subprocess.check_call(cmd)

def ensure_runtime():
    # Never import sklearn before survival stack install resolution.
    if _pkg_version("lightgbm") is None:
        _pip_install(["lightgbm"])

    if _pkg_version("scikit-survival") is None:
        _pip_install(["scikit-survival"])

    _purge_modules()
    try:
        import sklearn  # noqa: F401
        import sksurv   # noqa: F401
        print("[SETUP] survival stack import OK (native environment).")
        return
    except Exception as e:
        print("[SETUP] first survival import failed:", repr(e))

    # Hard fallback for ABI/version mismatch.
    # Python 3.12 is well-covered by sksurv 0.23.x + sklearn 1.5.x.
    # Python 3.13 is covered by sklearn 1.6.x and sksurv 0.24.x family.
    if sys.version_info[:2] <= (3, 12):
        pkgs = ["scikit-learn==1.5.2", "scikit-survival==0.23.1"]
    else:
        pkgs = ["scikit-learn==1.6.1", "scikit-survival==0.24.1"]

    _pip_install(pkgs)
    _purge_modules()

ensure_runtime()

import numpy as np
import pandas as pd
import lightgbm as lgb

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler

SKSURV_AVAILABLE = True
try:
    from sksurv.util import Surv
    from sksurv.ensemble import GradientBoostingSurvivalAnalysis, RandomSurvivalForest
    from sksurv.linear_model import CoxPHSurvivalAnalysis
except Exception as e:
    print("[WARN] sksurv unavailable after runtime fix:", repr(e))
    SKSURV_AVAILABLE = False

# ============================================================
# Config
# ============================================================

RUN_MODE = "full"
CV_BAG_TEST = True
STRAT_THR = 5000.0
POWER_CAL_24 = 1.10
USE_EXTERNAL_BLEND = True
HORIZONS_PRED = np.array([12.0, 24.0, 48.0, 72.0], dtype=float)

GBSA_SEEDS_FULL = (
    123, 456, 789, 777, 666, 1511, 1523, 2025, 2026, 2033,
    279, 239, 70, 77, 31, 2024, 2077, 3077, 123456, 654321,
    4640, 841, 7755, 8525, 2701, 8817, 8864, 4085, 8919, 934,
    4746, 1699, 7401, 7826, 4098, 2921, 1204, 2752, 8384, 1284,
)
COX_SEEDS_FULL  = (
    123, 456, 789, 777, 666, 1511, 1523, 2025, 2026, 2033,
    279, 239, 70, 77, 31, 2024, 2077, 3077, 123456, 654321,
)
RSF_SEEDS_FULL  = (
    123, 456, 789, 777, 666, 1511, 1523, 2025, 2026, 2033,
    279, 239, 70, 77, 31,
)
LGB_NEAR_SEEDS_FULL = (
    123, 456, 789, 777, 666, 1511, 1523, 2025, 2026, 2033,
    279, 239, 70, 77, 31, 2024, 2077, 3077, 123456, 654321,
    2034, 2035, 2036, 1984, 1991, 3255, 1011, 6241, 2790, 6847,
)
LGB_FAR_SEEDS_FULL = (
    8141, 7752, 432, 906, 6217, 7785, 1603, 7609, 965, 2506,
    3771, 7080, 4963, 7939, 2751, 473, 339, 3675, 5535, 4760,
    123, 456, 789, 777, 666, 1511, 1523, 2025, 2026, 2033,
)

GBSA_SEEDS_FAST = tuple(range(42, 52))
COX_SEEDS_FAST = (123, 456, 789)
RSF_SEEDS_FAST = (123, 456, 789)
LGB_NEAR_SEEDS_FAST = tuple(range(42, 52))
LGB_FAR_SEEDS_FAST = tuple(range(52, 62))

# Proven blend family
W_GBSA_NEAR_12 = 0.76; W_COX_NEAR_12 = 0.12; W_RSF_NEAR_12 = 0.02; W_LGB_NEAR_12 = 0.10
W_GBSA_NEAR_24 = 0.82; W_COX_NEAR_24 = 0.14; W_RSF_NEAR_24 = 0.02; W_LGB_NEAR_24 = 0.02
W_GBSA_NEAR_48 = 0.73; W_COX_NEAR_48 = 0.16; W_RSF_NEAR_48 = 0.03; W_LGB_NEAR_48 = 0.08

W_GBSA_FAR_24 = 0.62; W_COX_FAR_24 = 0.25; W_RSF_FAR_24 = 0.06; W_LGB_FAR_24 = 0.07
W_GBSA_FAR_48 = 0.35; W_COX_FAR_48 = 0.22; W_RSF_FAR_48 = 0.06; W_LGB_FAR_48 = 0.37

# ============================================================
# Paths
# ============================================================

def find_data_dir():
    candidates = [
        "/kaggle/input/competitions/WiDSWorldWide_GlobalDathon26",
        "/kaggle/input/WiDSWorldWide_GlobalDathon26",
        "/kaggle/input/widsworldwide-globaldathon26",
    ]
    for p in candidates:
        if os.path.exists(os.path.join(p, "train.csv")) and os.path.exists(os.path.join(p, "test.csv")):
            return p
    for p in glob.glob("/kaggle/input/**/train.csv", recursive=True):
        parent = os.path.dirname(p)
        if os.path.exists(os.path.join(parent, "test.csv")):
            return parent
    raise FileNotFoundError("Could not locate train.csv/test.csv in /kaggle/input")

DATA_DIR = find_data_dir()
WORK_DIR = "/kaggle/working" if os.path.exists("/kaggle/working") else "."
OUTPUT_PATH = os.path.join(WORK_DIR, "submission.csv")

train_df = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))
test_df = pd.read_csv(os.path.join(DATA_DIR, "test.csv"))
sample_sub = pd.read_csv(os.path.join(DATA_DIR, "sample_submission.csv"))

print("DATA_DIR:", DATA_DIR)
print("Train:", train_df.shape, "| Test:", test_df.shape)
print("Events:", train_df["event"].value_counts().to_dict())

# ============================================================
# Utils
# ============================================================

def clip01(x):
    return np.clip(np.asarray(x, dtype=float), 0.0, 1.0)

def enforce_monotonicity(preds):
    preds = clip01(preds)
    for j in range(1, preds.shape[1]):
        preds[:, j] = np.maximum(preds[:, j], preds[:, j - 1])
    return preds

def parse_score_from_filename(path: str):
    base = os.path.basename(path)
    m = re.search(r'[_-](0\.\d{3,6})(?:[^0-9]|$)', base)
    return float(m.group(1)) if m else None

def get_surv_predictions(model, X):
    surv_fns = model.predict_survival_function(X)
    preds = np.empty((len(surv_fns), len(HORIZONS_PRED)), dtype=float)
    for i, fn in enumerate(surv_fns):
        for j, h in enumerate(HORIZONS_PRED):
            try:
                preds[i, j] = 1.0 - float(fn(h))
            except Exception:
                try:
                    t_min, t_max = fn.domain
                    preds[i, j] = 1.0 - float(fn(np.clip(h, t_min, t_max)))
                except Exception:
                    xs = np.asarray(getattr(fn, "x", [0.0]), dtype=float)
                    ys = np.asarray(getattr(fn, "y", [1.0]), dtype=float)
                    idx = np.searchsorted(xs, h, side="right") - 1
                    s = 1.0 if idx < 0 else float(ys[idx])
                    preds[i, j] = 1.0 - s
    return clip01(preds)

def make_binary_target(time_vals, event_vals, horizon):
    unknown = (event_vals == 0) & (time_vals < horizon)
    y = ((event_vals == 1) & (time_vals <= horizon)).astype(float)
    return y, ~unknown

def compute_ipcw_weights(times, events, horizon):
    unique_t = np.sort(np.unique(times))
    surv = np.ones(len(unique_t), dtype=float)
    for i, t in enumerate(unique_t):
        at_risk = (times >= t).sum()
        censored_at_t = ((times == t) & (events == 0)).sum()
        if at_risk > 0:
            surv[i] = 1.0 - censored_at_t / at_risk
        if i > 0:
            surv[i] *= surv[i - 1]
    def G(t):
        idx = np.searchsorted(unique_t, t, side="right") - 1
        return max(surv[idx], 0.01) if idx >= 0 else 1.0
    weights = np.ones(len(times), dtype=float)
    for i in range(len(times)):
        if events[i] == 1 and times[i] <= horizon:
            weights[i] = 1.0 / G(times[i])
        elif times[i] >= horizon:
            weights[i] = 1.0 / G(horizon)
    return weights

def load_external_submissions(test_ids):
    if not USE_EXTERNAL_BLEND:
        return None, []
    need = ["event_id", "prob_12h", "prob_24h", "prob_48h", "prob_72h"]
    found = []
    seen = set()
    for path in glob.glob("/kaggle/input/**/*.csv", recursive=True):
        base = os.path.basename(path).lower()
        if "sample_submission" in base:
            continue
        if path in seen:
            continue
        seen.add(path)
        try:
            df = pd.read_csv(path)
        except Exception:
            continue
        if not set(need).issubset(df.columns):
            continue
        df = df[need].copy()
        if len(df) != len(test_ids):
            continue
        if df["event_id"].duplicated().any():
            continue
        if set(df["event_id"]) != set(test_ids):
            continue
        df = df.set_index("event_id").loc[test_ids].reset_index()
        arr = enforce_monotonicity(df[need[1:]].to_numpy())
        df.loc[:, need[1:]] = arr
        score = parse_score_from_filename(path)
        found.append((path, score, df))
    if not found:
        return None, []
    mats = []
    weights = []
    used = []
    for path, score, df in found:
        if score is None:
            w = 1.0
        else:
            w = max(0.25, (score - 0.955) * 100.0)
        mats.append(df[need[1:]].to_numpy())
        weights.append(w)
        used.append((path, score, w))
    weights = np.asarray(weights, dtype=float)
    weights = weights / weights.sum()
    blend = np.zeros_like(mats[0], dtype=float)
    for w, mat in zip(weights, mats):
        blend += w * mat
    ext = pd.DataFrame({"event_id": test_ids})
    ext[need[1:]] = blend
    ext.loc[:, need[1:]] = enforce_monotonicity(ext[need[1:]].to_numpy())
    return ext, used

# ============================================================
# Feature engineering
# ============================================================

def create_features(df: pd.DataFrame, fit_df: pd.DataFrame = None) -> pd.DataFrame:
    ref = fit_df if fit_df is not None else df
    result = df.copy()
    dist = result["dist_min_ci_0_5h"].clip(lower=1)
    speed = result["closing_speed_m_per_h"]
    perimeters = result["num_perimeters_0_5h"]
    area_first = result["area_first_ha"]

    result["log_distance"] = np.log1p(dist)
    result["inv_distance"] = 1 / (dist / 1000 + 0.1)
    result["inv_distance_sq"] = result["inv_distance"] ** 2
    result["sqrt_distance"] = np.sqrt(dist)
    result["dist_km"] = dist / 1000
    result["dist_km_sq"] = (dist / 1000) ** 2
    result["dist_km_cb"] = (dist / 1000) ** 3
    result["dist_rank"] = dist.rank(pct=True)

    fire_radius = np.sqrt(area_first * 10000 / np.pi)
    result["fire_radius_km"] = fire_radius / 1000
    result["radius_to_dist"] = fire_radius / dist
    result["area_to_dist_ratio"] = area_first / (dist / 1000 + 0.1)
    result["log_area_dist_ratio"] = np.log1p(area_first) - np.log1p(dist)

    result["has_movement"] = (perimeters > 1).astype(float)
    closing_pos = speed.clip(lower=0)
    result["eta_hours"] = np.where(closing_pos > 0.01, dist / closing_pos, 9999).clip(max=9999)
    result["log_eta"] = np.log1p(result["eta_hours"].clip(0, 9999))
    radial_growth = result["radial_growth_rate_m_per_h"].clip(lower=0)
    effective_closing = closing_pos + radial_growth
    result["effective_closing_speed"] = effective_closing
    result["eta_effective"] = np.where(effective_closing > 0.01, dist / effective_closing, 9999).clip(max=9999)
    result["threat_score"] = result["alignment_abs"] * speed / np.log1p(dist)
    result["threat_score_sq"] = result["threat_score"] ** 2
    result["fire_urgency"] = perimeters * speed
    result["growth_intensity"] = result["area_growth_rate_ha_per_h"] * perimeters

    result["zone_near"] = (dist < 5000).astype(float)
    result["zone_warning"] = ((dist >= 5000) & (dist < 10000)).astype(float)
    result["zone_far"] = (dist >= 10000).astype(float)

    ref_near_mask = ref["dist_min_ci_0_5h"].clip(lower=1) < 5000
    ref_far_mask = ~ref_near_mask
    near_speed_ref = ref.loc[ref_near_mask, "closing_speed_m_per_h"].values
    far_threat_ref = (
        ref.loc[ref_far_mask, "alignment_abs"]
        * ref.loc[ref_far_mask, "closing_speed_m_per_h"]
        / np.log1p(ref.loc[ref_far_mask, "dist_min_ci_0_5h"].clip(lower=1))
    ).values

    def rank_against_ref(vals, ref_vals):
        if len(ref_vals) == 0:
            return np.zeros(len(vals), dtype=float)
        return np.array([(ref_vals < v).mean() for v in vals], dtype=float)

    cur_near_mask = dist < 5000
    cur_far_mask = ~cur_near_mask
    near_speed_rank = np.zeros(len(result), dtype=float)
    far_threat_rank = np.zeros(len(result), dtype=float)
    if cur_near_mask.sum() > 0:
        near_speed_rank[cur_near_mask.values] = rank_against_ref(
            speed[cur_near_mask].values, near_speed_ref
        )
    if cur_far_mask.sum() > 0:
        threat_cur_far = (
            result.loc[cur_far_mask, "alignment_abs"]
            * speed[cur_far_mask]
            / np.log1p(dist[cur_far_mask])
        ).values
        far_threat_rank[cur_far_mask.values] = rank_against_ref(
            threat_cur_far, far_threat_ref
        )
    result["near_speed_rank"] = near_speed_rank
    result["far_threat_rank"] = far_threat_rank

    result["is_summer"] = result["event_start_month"].isin([6, 7, 8]).astype(float)
    result["is_afternoon"] = (
        (result["event_start_hour"] >= 12) & (result["event_start_hour"] < 20)
    ).astype(float)

    drop_cols = [
        "relative_growth_0_5h", "projected_advance_m",
        "centroid_displacement_m", "centroid_speed_m_per_h",
        "closing_speed_abs_m_per_h", "area_growth_abs_0_5h",
    ]
    result = result.drop(columns=[c for c in drop_cols if c in result.columns])
    result = result.replace([np.inf, -np.inf], np.nan).fillna(0)
    return result

train_processed = create_features(train_df, fit_df=train_df)
test_processed = create_features(test_df, fit_df=train_df)

event_values = train_df["event"].values
time_values = train_df["time_to_hit_hours"].values
dist_train = train_df["dist_min_ci_0_5h"].values
dist_test = test_df["dist_min_ci_0_5h"].values
near_train = dist_train < STRAT_THR
near_test = dist_test < STRAT_THR

# ============================================================
# Core model family
# ============================================================

def run_tri_survival_stack():
    y_surv = Surv.from_arrays(
        event=train_df["event"].astype(bool),
        time=train_df["time_to_hit_hours"]
    )

    # ----- GBSA -----
    X_gbsa_train = train_df.drop(columns=["event_id", "event", "time_to_hit_hours"])
    X_gbsa_test = test_df.drop(columns=["event_id"])

    gbsa_configs = [
        {"learning_rate":0.01,  "subsample":0.70, "max_depth":3, "min_samples_leaf":12, "min_samples_split":3, "n_estimators":1200, "dropout_rate":0.0},
        {"learning_rate":0.01,  "subsample":0.85, "max_depth":3, "min_samples_leaf":15, "min_samples_split":3, "n_estimators":1200, "dropout_rate":0.0},
        {"learning_rate":0.01,  "subsample":0.60, "max_depth":3, "min_samples_leaf":12, "min_samples_split":3, "n_estimators":1200, "dropout_rate":0.0},
        {"learning_rate":0.005, "subsample":0.85, "max_depth":3, "min_samples_leaf":12, "min_samples_split":3, "n_estimators":2000, "dropout_rate":0.0},
        {"learning_rate":0.01,  "subsample":0.85, "max_depth":3, "min_samples_leaf":20, "min_samples_split":3, "n_estimators":1400, "dropout_rate":0.0},
        {"learning_rate":0.008, "subsample":0.75, "max_depth":2, "min_samples_leaf":15, "min_samples_split":4, "n_estimators":1500, "dropout_rate":0.0},
        {"learning_rate":0.015, "subsample":0.70, "max_depth":3, "min_samples_leaf":10, "min_samples_split":3, "n_estimators":1000, "dropout_rate":0.0},
        {"learning_rate":0.005, "subsample":0.90, "max_depth":3, "min_samples_leaf":18, "min_samples_split":5, "n_estimators":2500, "dropout_rate":0.0},
        {"learning_rate":0.01,  "subsample":0.80, "max_depth":4, "min_samples_leaf":12, "min_samples_split":3, "n_estimators":1200, "dropout_rate":0.0},
        {"learning_rate":0.02,  "subsample":0.65, "max_depth":3, "min_samples_leaf":10, "min_samples_split":3, "n_estimators":800,  "dropout_rate":0.0},
    ]

    GBSA_SEEDS = GBSA_SEEDS_FULL if RUN_MODE == "full" else GBSA_SEEDS_FAST
    test_gbsa = np.zeros((len(X_gbsa_test), 4), dtype=float)

    print(f"GBSA: {len(gbsa_configs)} configs × {len(GBSA_SEEDS)} seeds × {'5-fold bag' if CV_BAG_TEST else 'full-fit'}")
    for cfg_idx, cfg in enumerate(gbsa_configs, 1):
        cfg_test = np.zeros((len(X_gbsa_test), 4), dtype=float)
        for seed in GBSA_SEEDS:
            if CV_BAG_TEST:
                seed_test = np.zeros((len(X_gbsa_test), 4), dtype=float)
                cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
                for tr_idx, _ in cv.split(X_gbsa_train, event_values):
                    m = GradientBoostingSurvivalAnalysis(**{**cfg, "random_state": seed})
                    m.fit(X_gbsa_train.iloc[tr_idx], y_surv[tr_idx])
                    seed_test += get_surv_predictions(m, X_gbsa_test) / 5.0
                cfg_test += seed_test / len(GBSA_SEEDS)
            else:
                mf = GradientBoostingSurvivalAnalysis(**{**cfg, "random_state": seed})
                mf.fit(X_gbsa_train, y_surv)
                cfg_test += get_surv_predictions(mf, X_gbsa_test) / len(GBSA_SEEDS)
        test_gbsa += cfg_test / len(gbsa_configs)
        print(f"  GBSA cfg {cfg_idx}/{len(gbsa_configs)} done")
    test_gbsa_raw = test_gbsa.copy()

    # ----- CoxPH -----
    COX_FEATURES_LIST_ENG = [
        "dist_km", "log_distance", "inv_distance",
        "closing_speed_m_per_h", "radial_growth_rate_m_per_h",
        "alignment_abs", "threat_score", "log_eta", "eta_effective",
        "area_to_dist_ratio", "fire_radius_km",
        "num_perimeters_0_5h", "has_movement",
        "near_speed_rank", "far_threat_rank",
        "is_summer", "is_afternoon",
        "zone_near", "zone_far",
    ]
    X_cox_train = train_processed[[c for c in COX_FEATURES_LIST_ENG if c in train_processed.columns]].copy()
    X_cox_test = test_processed[[c for c in COX_FEATURES_LIST_ENG if c in test_processed.columns]].copy()

    scaler = StandardScaler()
    X_cox_train_sc = pd.DataFrame(scaler.fit_transform(X_cox_train), columns=X_cox_train.columns, index=X_cox_train.index)
    X_cox_test_sc = pd.DataFrame(scaler.transform(X_cox_test), columns=X_cox_test.columns, index=X_cox_test.index)

    cox_alphas = [0.001, 0.01, 0.05, 0.10, 0.50, 1.00, 2.00]
    COX_SEEDS = COX_SEEDS_FULL if RUN_MODE == "full" else COX_SEEDS_FAST
    test_cox = np.zeros((len(X_cox_test_sc), 4), dtype=float)

    print(f"CoxPH: {len(cox_alphas)} alphas × {len(COX_SEEDS)} seeds × {'5-fold bag' if CV_BAG_TEST else 'full-fit'}")
    for alpha in cox_alphas:
        alpha_test = np.zeros((len(X_cox_test_sc), 4), dtype=float)
        for seed in COX_SEEDS:
            if CV_BAG_TEST:
                seed_test = np.zeros((len(X_cox_test_sc), 4), dtype=float)
                cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
                for tr_idx, _ in cv.split(X_cox_train_sc, event_values):
                    m = CoxPHSurvivalAnalysis(alpha=alpha)
                    try:
                        m.fit(X_cox_train_sc.iloc[tr_idx], y_surv[tr_idx])
                        seed_test += get_surv_predictions(m, X_cox_test_sc) / 5.0
                    except Exception:
                        seed_test += 0.5 / 5.0
                alpha_test += seed_test / len(COX_SEEDS)
            else:
                mf = CoxPHSurvivalAnalysis(alpha=alpha)
                try:
                    mf.fit(X_cox_train_sc, y_surv)
                    alpha_test += get_surv_predictions(mf, X_cox_test_sc) / len(COX_SEEDS)
                except Exception:
                    alpha_test += 0.5 / len(COX_SEEDS)
        test_cox += alpha_test / len(cox_alphas)
        print(f"  Cox alpha={alpha} done")

    # ----- RSF -----
    X_rsf_train = train_df.drop(columns=["event_id", "event", "time_to_hit_hours"])
    X_rsf_test = test_df.drop(columns=["event_id"])

    rsf_configs = [
        {"n_estimators": 200, "min_samples_leaf": 12, "max_features": "sqrt", "max_depth": None},
        {"n_estimators": 200, "min_samples_leaf": 18, "max_features": "sqrt", "max_depth": None},
        {"n_estimators": 200, "min_samples_leaf": 12, "max_features": 0.5, "max_depth": 5},
    ]
    RSF_SEEDS = RSF_SEEDS_FULL if RUN_MODE == "full" else RSF_SEEDS_FAST
    test_rsf = np.zeros((len(X_rsf_test), 4), dtype=float)

    print(f"RSF: {len(rsf_configs)} configs × {len(RSF_SEEDS)} seeds × {'5-fold bag' if CV_BAG_TEST else 'full-fit'}")
    for cfg_idx, cfg in enumerate(rsf_configs, 1):
        cfg_test = np.zeros((len(X_rsf_test), 4), dtype=float)
        for seed in RSF_SEEDS:
            if CV_BAG_TEST:
                seed_test = np.zeros((len(X_rsf_test), 4), dtype=float)
                cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
                for tr_idx, _ in cv.split(X_rsf_train, event_values):
                    m = RandomSurvivalForest(**{**cfg, "random_state": seed, "n_jobs": -1})
                    m.fit(X_rsf_train.iloc[tr_idx], y_surv[tr_idx])
                    seed_test += get_surv_predictions(m, X_rsf_test) / 5.0
                cfg_test += seed_test / len(RSF_SEEDS)
            else:
                mf = RandomSurvivalForest(**{**cfg, "random_state": seed, "n_jobs": -1})
                mf.fit(X_rsf_train, y_surv)
                cfg_test += get_surv_predictions(mf, X_rsf_test) / len(RSF_SEEDS)
        test_rsf += cfg_test / len(rsf_configs)
        print(f"  RSF cfg {cfg_idx}/{len(rsf_configs)} done")

    # ----- Zone-split LGB -----
    NEAR_LGB_FEATURES = [
        "closing_speed_m_per_h", "radial_growth_rate_m_per_h",
        "alignment_abs", "num_perimeters_0_5h", "area_growth_rate_ha_per_h",
        "eta_effective", "log_eta", "dist_km", "threat_score",
        "near_speed_rank", "event_start_hour", "is_afternoon", "fire_urgency",
        "area_first_ha", "fire_radius_km",
    ]
    FAR_LGB_FEATURES = [
        "dist_km", "log_distance", "inv_distance",
        "closing_speed_m_per_h", "alignment_abs",
        "threat_score", "log_eta", "eta_effective",
        "area_to_dist_ratio", "num_perimeters_0_5h",
        "far_threat_rank", "is_summer", "zone_far",
    ]

    avail_near_lgb = [f for f in NEAR_LGB_FEATURES if f in train_processed.columns]
    avail_far_lgb = [f for f in FAR_LGB_FEATURES if f in train_processed.columns]

    X_near_lgb_train = train_processed[avail_near_lgb]
    X_near_lgb_test = test_processed[[f for f in avail_near_lgb if f in test_processed.columns]]
    X_far_lgb_train = train_processed[avail_far_lgb]
    X_far_lgb_test = test_processed[[f for f in avail_far_lgb if f in test_processed.columns]]

    near_lgb_cfgs = {
        24: {"max_depth":2, "learning_rate":0.04, "n_estimators":200,
             "subsample":0.8, "colsample_bytree":0.8, "min_child_samples":4,
             "reg_alpha":0.5, "reg_lambda":1.5, "num_leaves":4},
        48: {"max_depth":2, "learning_rate":0.05, "n_estimators":150,
             "subsample":0.8, "colsample_bytree":0.8, "min_child_samples":3,
             "reg_alpha":0.3, "reg_lambda":1.0, "num_leaves":4},
    }
    far_lgb_cfgs = {
        24: {"max_depth":2, "learning_rate":0.03, "n_estimators":200,
             "subsample":0.7, "colsample_bytree":0.7, "min_child_samples":8,
             "reg_alpha":1.0, "reg_lambda":3.0, "num_leaves":4},
        48: {"max_depth":2, "learning_rate":0.05, "n_estimators":150,
             "subsample":0.8, "colsample_bytree":0.8, "min_child_samples":6,
             "reg_alpha":0.5, "reg_lambda":2.0, "num_leaves":4},
    }

    LGB_NEAR_SEEDS = LGB_NEAR_SEEDS_FULL if RUN_MODE == "full" else LGB_NEAR_SEEDS_FAST
    LGB_FAR_SEEDS = LGB_FAR_SEEDS_FULL if RUN_MODE == "full" else LGB_FAR_SEEDS_FAST
    lgb_near_test, lgb_far_test = {}, {}

    print(f"Zone-LGB: near {len(LGB_NEAR_SEEDS)} seeds | far {len(LGB_FAR_SEEDS)} seeds")
    for horizon in [24, 48]:
        y_bin, mask = make_binary_target(time_values, event_values, horizon)
        valid_idx = np.where(mask)[0]

        cfg = near_lgb_cfgs[horizon]
        all_test = np.zeros(len(X_near_lgb_test), dtype=float)
        for seed in LGB_NEAR_SEEDS:
            ipcw_w = compute_ipcw_weights(time_values[valid_idx], event_values[valid_idx], horizon)
            mf = lgb.LGBMClassifier(**cfg, objective="binary", random_state=seed, verbose=-1)
            mf.fit(X_near_lgb_train.iloc[valid_idx], y_bin[valid_idx], sample_weight=ipcw_w)
            all_test += mf.predict_proba(X_near_lgb_test)[:, 1]
        lgb_near_test[horizon] = all_test / len(LGB_NEAR_SEEDS)

        cfg = far_lgb_cfgs[horizon]
        all_test = np.zeros(len(X_far_lgb_test), dtype=float)
        for seed in LGB_FAR_SEEDS:
            ipcw_w = compute_ipcw_weights(time_values[valid_idx], event_values[valid_idx], horizon)
            mf = lgb.LGBMClassifier(**cfg, objective="binary", random_state=seed, verbose=-1)
            mf.fit(X_far_lgb_train.iloc[valid_idx], y_bin[valid_idx], sample_weight=ipcw_w)
            all_test += mf.predict_proba(X_far_lgb_test)[:, 1]
        lgb_far_test[horizon] = all_test / len(LGB_FAR_SEEDS)
        print(f"  LGB horizon={horizon} done")

    # ----- Blend -----
    test_gbsa = test_gbsa_raw.copy()
    if POWER_CAL_24 != 1.0:
        test_gbsa[:, 1] = np.clip(test_gbsa[:, 1] ** POWER_CAL_24, 0, 1)

    def blend_zone(gbsa, cox, rsf, lgb_p, wg, wc, wr, wl):
        return wg * gbsa + wc * cox + wr * rsf + wl * lgb_p

    test_blend = np.zeros_like(test_gbsa)
    test_blend[:, 0] = np.where(
        near_test,
        blend_zone(test_gbsa[:,0], test_cox[:,0], test_rsf[:,0], lgb_near_test[24],
                   W_GBSA_NEAR_12, W_COX_NEAR_12, W_RSF_NEAR_12, W_LGB_NEAR_12),
        np.clip(test_gbsa[:,0], 0.0, 0.10)
    )
    test_blend[:, 1] = np.where(
        near_test,
        blend_zone(test_gbsa[:,1], test_cox[:,1], test_rsf[:,1], lgb_near_test[24],
                   W_GBSA_NEAR_24, W_COX_NEAR_24, W_RSF_NEAR_24, W_LGB_NEAR_24),
        blend_zone(test_gbsa[:,1], test_cox[:,1], test_rsf[:,1], lgb_far_test[24],
                   W_GBSA_FAR_24, W_COX_FAR_24, W_RSF_FAR_24, W_LGB_FAR_24)
    )
    test_blend[:, 2] = np.where(
        near_test,
        blend_zone(test_gbsa[:,2], test_cox[:,2], test_rsf[:,2], lgb_near_test[48],
                   W_GBSA_NEAR_48, W_COX_NEAR_48, W_RSF_NEAR_48, W_LGB_NEAR_48),
        blend_zone(test_gbsa[:,2], test_cox[:,2], test_rsf[:,2], lgb_far_test[48],
                   W_GBSA_FAR_48, W_COX_FAR_48, W_RSF_FAR_48, W_LGB_FAR_48)
    )
    test_blend[:, 3] = 1.0

    test_final = enforce_monotonicity(test_blend)
    sub = pd.DataFrame({
        "event_id": test_df["event_id"].values,
        "prob_12h": test_final[:, 0],
        "prob_24h": test_final[:, 1],
        "prob_48h": test_final[:, 2],
        "prob_72h": test_final[:, 3],
    })
    return sub

def run_fallback_lgb_only():
    print("[FALLBACK] running pure LightGBM zone experts")
    near_mask_test = near_test
    NEAR_LGB_FEATURES = [
        "closing_speed_m_per_h", "radial_growth_rate_m_per_h",
        "alignment_abs", "num_perimeters_0_5h", "area_growth_rate_ha_per_h",
        "eta_effective", "log_eta", "dist_km", "threat_score",
        "near_speed_rank", "event_start_hour", "is_afternoon", "fire_urgency",
        "area_first_ha", "fire_radius_km",
    ]
    FAR_LGB_FEATURES = [
        "dist_km", "log_distance", "inv_distance",
        "closing_speed_m_per_h", "alignment_abs",
        "threat_score", "log_eta", "eta_effective",
        "area_to_dist_ratio", "num_perimeters_0_5h",
        "far_threat_rank", "is_summer", "zone_far",
    ]
    near_feats = [f for f in NEAR_LGB_FEATURES if f in train_processed.columns]
    far_feats = [f for f in FAR_LGB_FEATURES if f in train_processed.columns]

    near_cfgs = {
        12: {"max_depth":2, "learning_rate":0.04, "n_estimators":220,
             "subsample":0.8, "colsample_bytree":0.8, "min_child_samples":4,
             "reg_alpha":0.5, "reg_lambda":1.5, "num_leaves":4},
        24: {"max_depth":2, "learning_rate":0.04, "n_estimators":200,
             "subsample":0.8, "colsample_bytree":0.8, "min_child_samples":4,
             "reg_alpha":0.5, "reg_lambda":1.5, "num_leaves":4},
        48: {"max_depth":2, "learning_rate":0.05, "n_estimators":160,
             "subsample":0.8, "colsample_bytree":0.8, "min_child_samples":3,
             "reg_alpha":0.3, "reg_lambda":1.0, "num_leaves":4},
    }
    far_cfgs = {
        24: {"max_depth":2, "learning_rate":0.03, "n_estimators":220,
             "subsample":0.7, "colsample_bytree":0.7, "min_child_samples":8,
             "reg_alpha":1.0, "reg_lambda":3.0, "num_leaves":4},
        48: {"max_depth":2, "learning_rate":0.05, "n_estimators":160,
             "subsample":0.8, "colsample_bytree":0.8, "min_child_samples":6,
             "reg_alpha":0.5, "reg_lambda":2.0, "num_leaves":4},
    }

    out = pd.DataFrame({"event_id": test_df["event_id"].values})
    for horizon in [12, 24, 48]:
        y_bin, mask = make_binary_target(time_values, event_values, horizon)
        valid_idx = np.where(mask)[0]

        cfg = near_cfgs[horizon]
        p_near = np.zeros(len(test_df), dtype=float)
        near_model = lgb.LGBMClassifier(**cfg, objective="binary", random_state=123, verbose=-1)
        w = compute_ipcw_weights(time_values[valid_idx], event_values[valid_idx], horizon)
        near_model.fit(train_processed.iloc[valid_idx][near_feats], y_bin[valid_idx], sample_weight=w)
        p_near = near_model.predict_proba(test_processed[near_feats])[:, 1]

        if horizon == 12:
            p = np.where(near_mask_test, p_near, 0.0)
        else:
            cfg = far_cfgs[horizon]
            far_model = lgb.LGBMClassifier(**cfg, objective="binary", random_state=456 + horizon, verbose=-1)
            w = compute_ipcw_weights(time_values[valid_idx], event_values[valid_idx], horizon)
            far_model.fit(train_processed.iloc[valid_idx][far_feats], y_bin[valid_idx], sample_weight=w)
            p_far = far_model.predict_proba(test_processed[far_feats])[:, 1]
            p = np.where(near_mask_test, p_near, p_far)
        out[f"prob_{horizon}h"] = clip01(p)

    out["prob_72h"] = 1.0
    out.loc[:, ["prob_12h", "prob_24h", "prob_48h", "prob_72h"]] = enforce_monotonicity(
        out[["prob_12h", "prob_24h", "prob_48h", "prob_72h"]].to_numpy()
    )
    return out

# ============================================================
# Main
# ============================================================

if SKSURV_AVAILABLE:
    submission = run_tri_survival_stack()
else:
    submission = run_fallback_lgb_only()

submission = sample_sub[["event_id"]].merge(submission, on="event_id", how="left")
submission.loc[:, ["prob_12h", "prob_24h", "prob_48h", "prob_72h"]] = enforce_monotonicity(
    submission[["prob_12h", "prob_24h", "prob_48h", "prob_72h"]].to_numpy()
)

external_blend, used_external = load_external_submissions(sample_sub["event_id"].tolist())
if external_blend is not None and len(used_external) > 0:
    parsed_scores = [s for _, s, _ in used_external if s is not None]
    max_score = max(parsed_scores) if parsed_scores else None
    if max_score is not None and max_score >= 0.9715:
        ext_w = 0.18
    elif max_score is not None and max_score >= 0.9705:
        ext_w = 0.12
    else:
        ext_w = 0.08

    print("[External Blend] files used:")
    for path, score, w in used_external[:20]:
        print(" -", os.path.basename(path), "| score_hint=", score, "| weight=", round(w, 4))
    print(f"[External Blend] blend_weight={ext_w:.2f}")

    cols = ["prob_12h", "prob_24h", "prob_48h", "prob_72h"]
    for c in cols:
        submission[c] = (1.0 - ext_w) * submission[c].to_numpy() + ext_w * external_blend[c].to_numpy()
    submission.loc[:, cols] = enforce_monotonicity(submission[cols].to_numpy())

submission = submission[["event_id", "prob_12h", "prob_24h", "prob_48h", "prob_72h"]].copy()
assert submission["event_id"].tolist() == sample_sub["event_id"].tolist(), "event_id order mismatch"
assert not submission["event_id"].duplicated().any(), "duplicate event_id"
for c in ["prob_12h", "prob_24h", "prob_48h", "prob_72h"]:
    assert np.isfinite(submission[c]).all(), f"non-finite in {c}"
    assert ((submission[c] >= 0) & (submission[c] <= 1)).all(), f"{c} out of [0,1]"

assert (submission["prob_12h"] <= submission["prob_24h"] + 1e-12).all()
assert (submission["prob_24h"] <= submission["prob_48h"] + 1e-12).all()
assert (submission["prob_48h"] <= submission["prob_72h"] + 1e-12).all()

submission.to_csv(OUTPUT_PATH, index=False)
print("\nSaved:", OUTPUT_PATH)
print(submission.head())
print("\nSummary:")
print(submission[["prob_12h", "prob_24h", "prob_48h", "prob_72h"]].describe().T[["mean", "std", "min", "max"]])


[SETUP] pip install: scikit-survival
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.0/4.0 MB 169.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 260.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 222.1/222.1 kB 291.1 MB/s eta 0:00:00
[SETUP] survival stack import OK (native environment).
DATA_DIR: /kaggle/input/competitions/WiDSWorldWide_GlobalDathon26
Train: (221, 37) | Test: (95, 35)
Events: {0: 152, 1: 69}
GBSA: 10 configs × 40 seeds × 5-fold bag
  GBSA cfg 1/10 done
  GBSA cfg 2/10 done
  GBSA cfg 3/10 done
  GBSA cfg 4/10 done
  GBSA cfg 5/10 done
  GBSA cfg 6/10 done
  GBSA cfg 7/10 done
  GBSA cfg 8/10 done
  GBSA cfg 9/10 done
  GBSA cfg 10/10 done
CoxPH: 7 alphas × 20 seeds × 5-fold bag
  Cox alpha=0.001 done
  Cox alpha=0.01 done
  Cox alpha=0.05 done
  Cox alpha=0.1 done
  Cox alpha=0.5 done
  Cox alpha=1.0 done
  Cox alpha=2.0 done
RSF: 3 configs × 15 seeds × 5-fold bag
  RSF cfg 1/3 done
  RSF cfg 2/3 done
  RSF cfg 3/3 done
Zone-LGB